In [0]:
# Databricks Notebook: BEL Pipeline Visual Verification
# Run each cell separately in Databricks Notebook (Python cluster, not SQL Warehouse)

# ============================================================
# CELL 1: Setup & Data Loading
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800', '#607D8B', '#E91E63', '#00BCD4']

# Load data from Delta tables
df_inforce = spark.sql("""
    SELECT cohort_id, sex, scenario_id, projection_month, inforce_open, policy_count
    FROM workspace.life_insurance_bel_int.int_inforce_rollforward
    WHERE scenario_id = 'BASE'
""").toPandas()

df_discount = spark.sql("""
    SELECT DISTINCT scenario_id, projection_month, discount_factor, cashflow_type
    FROM workspace.life_insurance_bel_int.int_cashflows_discounted
    WHERE cashflow_type = 'death_benefit'
      AND cohort_id = 'C1' AND sex = 'M'
      AND scenario_id IN ('BASE', 'RATE_UP', 'RATE_DOWN')
""").toPandas()

df_bel = spark.sql("""
    SELECT cohort_id, sex, scenario_id, premium_pv, benefit_pv, expense_pv, bel_amount, bel_per_policy
    FROM workspace.life_insurance_bel_int.int_bel_components
""").toPandas()

df_sensitivity = spark.sql("""
    SELECT cohort_id, sex, scenario_id, scenario_group, bel_base, bel_stressed, delta_bel, delta_bel_pct
    FROM workspace.life_insurance_bel_mart.mart_bel_sensitivity
""").toPandas()

df_portfolio = spark.sql("""
    SELECT scenario_id, scenario_group, total_bel, total_delta_bel, weighted_avg_delta_pct
    FROM workspace.life_insurance_bel_mart.mart_portfolio_overview
    ORDER BY total_delta_bel DESC
""").toPandas()

print("Data loaded successfully.")
print(f"  Inforce rows: {len(df_inforce):,}")
print(f"  BEL rows: {len(df_bel):,}")
print(f"  Sensitivity rows: {len(df_sensitivity):,}")


# ============================================================
# CELL 2: Survival Curve — In-Force Decay by Cohort
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('In-Force Survival Curves by Cohort (Base Scenario)', fontsize=16, fontweight='bold')

cohorts = ['C1', 'C2', 'C3', 'C4']
cohort_info = {
    'C1': 'Issue 35, Attained 40, 25Y remaining',
    'C2': 'Issue 50, Attained 60, 15Y remaining',
    'C3': 'Issue 60, Attained 65, 15Y remaining',
    'C4': 'Issue 65, Attained 68, 12Y remaining'
}

for idx, cohort in enumerate(cohorts):
    ax = axes[idx // 2][idx % 2]
    cohort_data = df_inforce[df_inforce['cohort_id'] == cohort]

    for i, sex in enumerate(['M', 'F']):
        sex_data = cohort_data[cohort_data['sex'] == sex].sort_values('projection_month')
        survival_pct = sex_data['inforce_open'] / sex_data['policy_count'].iloc[0] * 100
        color = COLORS[i]
        label = f"{'Male' if sex == 'M' else 'Female'} (n={sex_data['policy_count'].iloc[0]:,.0f})"
        ax.plot(sex_data['projection_month'] / 12, survival_pct, color=color, linewidth=2, label=label)

    ax.set_title(f"Cohort {cohort}: {cohort_info[cohort]}", fontsize=11)
    ax.set_xlabel('Projection Year')
    ax.set_ylabel('In-Force (%)')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)

plt.tight_layout()
plt.savefig('/tmp/survival_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Survival curves: older cohorts decay faster (higher mortality + lapse)")


# ============================================================
# CELL 3: Discount Factor Curves — Base vs Stressed
# ============================================================

fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle('Discount Factor Curves: Base vs Rate Stress (C1-M, Death Benefit)', fontsize=14, fontweight='bold')

scenario_styles = {
    'BASE': {'color': '#2196F3', 'linewidth': 2.5, 'linestyle': '-'},
    'RATE_UP': {'color': '#4CAF50', 'linewidth': 1.8, 'linestyle': '--'},
    'RATE_DOWN': {'color': '#FF5722', 'linewidth': 1.8, 'linestyle': '--'}
}

for scenario_id, style in scenario_styles.items():
    data = df_discount[df_discount['scenario_id'] == scenario_id].sort_values('projection_month')
    ax.plot(data['projection_month'] / 12, data['discount_factor'],
            label=scenario_id, **style)

ax.set_xlabel('Projection Year', fontsize=12)
ax.set_ylabel('Discount Factor', fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)

# Annotate spread
ax.annotate('Rate Up:\nhigher discounting\n→ lower BEL',
            xy=(15, 0.55), fontsize=9, color='#4CAF50', fontstyle='italic')
ax.annotate('Rate Down:\nlower discounting\n→ higher BEL',
            xy=(15, 0.75), fontsize=9, color='#FF5722', fontstyle='italic')

plt.tight_layout()
plt.savefig('/tmp/discount_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Discount curves: RATE_UP curve below BASE, RATE_DOWN above — as expected")


# ============================================================
# CELL 4: BEL Waterfall — PV Component Decomposition (Base)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('BEL Waterfall: PV Component Decomposition (Base Scenario)', fontsize=14, fontweight='bold')

base = df_bel[df_bel['scenario_id'] == 'BASE'].copy()

# Left: Total BEL by cohort × sex
ax = axes[0]
base_sorted = base.sort_values('bel_amount', ascending=True)
labels = [f"{r['cohort_id']}-{r['sex']}" for _, r in base_sorted.iterrows()]
premium_vals = base_sorted['premium_pv'].values / 1e6
benefit_vals = base_sorted['benefit_pv'].values / 1e6
expense_vals = base_sorted['expense_pv'].values / 1e6

y_pos = np.arange(len(labels))
bar_height = 0.25

ax.barh(y_pos - bar_height, premium_vals, bar_height, label='Premium PV (inflow)', color='#2196F3', alpha=0.8)
ax.barh(y_pos, benefit_vals, bar_height, label='Benefit PV (outflow)', color='#FF5722', alpha=0.8)
ax.barh(y_pos + bar_height, expense_vals, bar_height, label='Expense PV (outflow)', color='#FF9800', alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Present Value (€M)')
ax.set_title('PV Components by Model Point')
ax.legend(fontsize=9, loc='lower right')
ax.axvline(x=0, color='black', linewidth=0.5)

# Right: BEL per policy by cohort × sex
ax2 = axes[1]
base_pp = base.sort_values('bel_per_policy', ascending=True)
labels_pp = [f"{r['cohort_id']}-{r['sex']}" for _, r in base_pp.iterrows()]
bel_pp_vals = base_pp['bel_per_policy'].values

colors_pp = ['#4CAF50' if v < 0 else '#FF5722' for v in bel_pp_vals]
ax2.barh(labels_pp, bel_pp_vals, color=colors_pp, alpha=0.8)
ax2.set_xlabel('BEL per Policy (€)')
ax2.set_title('BEL per Policy (negative = profitable)')
ax2.axvline(x=0, color='black', linewidth=0.8)

for i, v in enumerate(bel_pp_vals):
    ax2.text(v + (50 if v >= 0 else -50), i, f'{v:,.0f}',
             va='center', ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/bel_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Waterfall: C3 highest BEL (senior, high mortality), C1 lowest (young, premium-dominated)")


# ============================================================
# CELL 5: Stress Delta Plot — Scenario Sensitivity
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('BEL Sensitivity Analysis', fontsize=14, fontweight='bold')

# Left: Portfolio-level delta by scenario
ax = axes[0]
port = df_portfolio[df_portfolio['scenario_id'] != 'BASE'].sort_values('total_delta_bel')
colors_port = ['#4CAF50' if v < 0 else '#FF5722' for v in port['total_delta_bel']]

ax.barh(port['scenario_id'], port['total_delta_bel'] / 1e6, color=colors_port, alpha=0.85)
ax.set_xlabel('Delta BEL (€M)')
ax.set_title('Portfolio-Level BEL Change by Scenario')
ax.axvline(x=0, color='black', linewidth=0.8)

for i, (_, row) in enumerate(port.iterrows()):
    pct = row['weighted_avg_delta_pct'] * 100
    val = row['total_delta_bel'] / 1e6
    ax.text(val + (0.3 if val >= 0 else -0.3), i,
            f'{pct:+.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

# Right: Heatmap — delta_bel_pct by cohort-sex × scenario
ax2 = axes[1]

# Pivot for heatmap
heat_data = df_sensitivity.copy()
heat_data['model_point'] = heat_data['cohort_id'] + '-' + heat_data['sex']
heat_pivot = heat_data.pivot_table(index='model_point', columns='scenario_id', values='delta_bel_pct')

# Order scenarios
scenario_order = ['MORT_DOWN', 'LAPSE_UP', 'RATE_UP', 'LAPSE_DOWN', 'RATE_DOWN', 'PERSIST_IMPROVE', 'MORT_UP', 'COMBINED_ADVERSE']
scenario_order = [s for s in scenario_order if s in heat_pivot.columns]
heat_pivot = heat_pivot[scenario_order]

# Order model points
mp_order = ['C1-M', 'C1-F', 'C2-M', 'C2-F', 'C3-M', 'C3-F', 'C4-M', 'C4-F']
mp_order = [m for m in mp_order if m in heat_pivot.index]
heat_pivot = heat_pivot.loc[mp_order]

im = ax2.imshow(heat_pivot.values * 100, cmap='RdYlGn_r', aspect='auto',
                vmin=-100, vmax=100)
ax2.set_xticks(range(len(scenario_order)))
ax2.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=8)
ax2.set_yticks(range(len(mp_order)))
ax2.set_yticklabels(mp_order, fontsize=9)
ax2.set_title('Delta BEL % by Model Point × Scenario')

# Add text annotations
for i in range(len(mp_order)):
    for j in range(len(scenario_order)):
        val = heat_pivot.values[i, j] * 100
        color = 'white' if abs(val) > 50 else 'black'
        ax2.text(j, i, f'{val:+.0f}%', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax2, label='Delta BEL (%)', shrink=0.8)

plt.tight_layout()
plt.savefig('/tmp/stress_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Sensitivity: MORT_UP dominant risk; C1 shows opposite lapse direction vs C3/C4")

In [0]:
# ============================================================
# CELL 6: Export All Assets for Medium Article
# ============================================================

import os

# 저장 경로 (Workspace Files)
OUTPUT_DIR = "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/assets"

# --- 1. 시각화 4장 다시 저장 (이미 plt로 만든 것들) ---
# 이미 /tmp/에 저장됐으니 복사
import shutil
for fname in ['survival_curves.png', 'discount_curves.png', 'bel_waterfall.png', 'stress_delta.png']:
    src = f'/tmp/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUTPUT_DIR}/{fname}')
        print(f"✓ Copied {fname}")

# --- 2. 쿼리 결과 CSV 저장 ---

queries = {
    "01_bel_base_results": """
        SELECT cohort_id, sex, issue_age, remaining_term_years,
               ROUND(premium_pv, 0) as premium_pv,
               ROUND(benefit_pv, 0) as benefit_pv,
               ROUND(expense_pv, 0) as expense_pv,
               ROUND(bel_amount, 0) as bel_amount,
               policy_count,
               ROUND(bel_per_policy, 0) as bel_per_policy
        FROM workspace.life_insurance_bel_mart.mart_bel_components_core
        WHERE scenario_id = 'BASE'
        ORDER BY bel_per_policy DESC
    """,
    "02_portfolio_overview": """
        SELECT scenario_id,
               COALESCE(scenario_group, 'Base') as scenario_group,
               ROUND(total_bel / 1000000, 2) as total_bel_M,
               ROUND(total_delta_bel / 1000000, 2) as delta_bel_M,
               ROUND(weighted_avg_delta_pct * 100, 1) as delta_pct
        FROM workspace.life_insurance_bel_mart.mart_portfolio_overview
        ORDER BY total_delta_bel DESC
    """,
    "03_validation_summary": """
        SELECT overall_validation_status, COUNT(*) as count
        FROM workspace.life_insurance_bel_mart.mart_validation_summary
        GROUP BY 1
    """,
    "04_sensitivity_by_cohort": """
        SELECT cohort_id, sex, scenario_id,
               ROUND(delta_bel / 1000000, 2) as delta_M,
               ROUND(delta_bel_pct * 100, 1) as delta_pct
        FROM workspace.life_insurance_bel_mart.mart_bel_sensitivity
        WHERE scenario_id IN ('MORT_UP', 'RATE_UP', 'LAPSE_UP')
        ORDER BY scenario_id, cohort_id, sex
    """,
    "05_inforce_decay": """
        SELECT cohort_id, sex,
               MAX(CASE WHEN projection_month = 1 THEN ROUND(inforce_open, 0) END) as month_1,
               MAX(CASE WHEN projection_month = 60 THEN ROUND(inforce_open, 0) END) as year_5,
               MAX(CASE WHEN projection_month = 120 THEN ROUND(inforce_open, 0) END) as year_10,
               MAX(CASE WHEN projection_month = 180 THEN ROUND(inforce_open, 0) END) as year_15
        FROM workspace.life_insurance_bel_int.int_inforce_rollforward
        WHERE scenario_id = 'BASE'
        GROUP BY cohort_id, sex
        ORDER BY cohort_id, sex
    """
}

for name, query in queries.items():
    df = spark.sql(query).toPandas()
    path = f"{OUTPUT_DIR}/{name}.csv"
    df.to_csv(path, index=False)
    print(f"✓ Saved {name}.csv ({len(df)} rows)")

# --- 3. 저장된 파일 목록 확인 ---
print(f"\n📁 All files in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  {f} ({size:,} bytes)")